In [22]:
path_hybrid_fusion_retrieval = "../results/qid2nctids_results_ollama:phi4_mstro_k20_bm25wt1_medcptwt1_N2000.json"

dataset = "mstro"
path_corpus = f"../dataset/{dataset}/corpus.jsonl"
path_id2queries = f"../dataset/{dataset}/id2queries.json"


In [23]:
import json


# Define a function to map corpus items to the desired structure
def map_corpus_item(item):
    mapped_item = {}
    
    # Map _id to NCTID
    if '_id' in item:
        mapped_item['NCTID'] = item['_id']
    
    # Keep title and text as they are
    if 'title' in item:
        mapped_item['title'] = item['title']
    if 'text' in item:
        mapped_item['text'] = item['text']
    
    # Map metadata fields
    if 'metadata' in item:
        metadata = item['metadata']
        for field in ['brief_title', 'phase', 'drugs', 'drugs_list', 'diseases', 
                     'diseases_list', 'enrollment', 'inclusion_criteria', 
                     'exclusion_criteria', 'brief_summary']:
            if field in metadata:
                mapped_item[field] = metadata[field]
    
    return mapped_item

# Open and load the JSON files
with open(path_id2queries, 'r') as file:
    id2queries = json.load(file)

with open(path_hybrid_fusion_retrieval, 'r') as file:
    hybrid_fusion_retrieval = json.load(file)

# Load JSONL corpus file - this contains the full trial information
corpus = {}
with open(path_corpus, 'r') as file:
    for line in file:
        # Parse each line as a separate JSON object
        item = json.loads(line)
        # Use the _id field as the key
        if '_id' in item:
            # Store the original item for mapping later
            corpus[item['_id']] = item
        else:
            print(f"Warning: Item without _id in corpus file: {line[:100]}...")

print(f"Loaded {len(corpus)} trials from corpus")

# Create the retrieved_trials data structure
retrieved_trials = []
missing_trials = 0

for patient_id in id2queries:
    entry = id2queries[patient_id]
    
    new_item = {
        "patient_id": patient_id,
        "patient": entry["raw"],
        "0": []  # Initialize as an empty list to hold all retrieved trials
    }

    # Check if this patient has any retrieved trials
    if patient_id in hybrid_fusion_retrieval:
        trial_ids = hybrid_fusion_retrieval[patient_id]
        
        # For each trial ID, get the full trial information from corpus
        for trial_id in trial_ids:
            if trial_id in corpus:
                # Apply the mapping and add the mapped information
                mapped_trial = map_corpus_item(corpus[trial_id])
                new_item["0"].append(mapped_trial)
            else:
                print(f"Warning: Trial ID {trial_id} not found in corpus for patient {patient_id}")
                missing_trials += 1
    else:
        print(f"Warning: No retrieved trials for patient {patient_id}")
    
    retrieved_trials.append(new_item)

print(f"Created retrieved_trials with {len(retrieved_trials)} patients")
print(f"Missing trials: {missing_trials}")

# Save the retrieved trials to a file
path_trials = f"../dataset/{dataset}/retrieved_trials.json"
with open(path_trials, 'w', encoding='utf-8') as file:
    json.dump(retrieved_trials, file, ensure_ascii=False, indent=4)

print(f"Saved retrieved trials to {path_trials}")

Loaded 19 trials from corpus
Created retrieved_trials with 20 patients
Missing trials: 0
Saved retrieved trials to ../dataset/mstro/retrieved_trials.json
